# 08.1 — Folder hierarchy generation

When a run finishes, where do its results go? In this pipeline the answer is a **19-level-deep folder path** where *every level encodes a piece of the configuration* — model type, learning rate, augmentation strengths, loss weights, the curriculum regime, and finally the fold number. The path *is* the experiment record: you can read the entire config off it, and MATLAB's results aggregator discovers and groups runs by *walking this tree* (Critical Note #15). The port reproduces the hierarchy byte-for-byte so Python-trained runs land exactly where the MATLAB tooling expects them. This notebook is that folder generator.

This notebook covers:

* Why encode the whole config into a directory path.
* The 19-level hierarchy, generated live.
* `build_matlab_run_dirs` and the three directories it returns.
* Why byte-exact parity with MATLAB's naming matters for discovery.

**Prerequisites:** [00.6 git & GitHub](../00_orientation/00.6_git_and_github_for_matlab_users.ipynb) (paths), [07.6 the curriculum walkthrough](../07_dynamic_curriculum/07.6_walkthrough_soft_three_stage_curriculum_shortened.ipynb) (the config it encodes).

## Section 1 — What MATLAB does

`cgg_generateDecodingFolders` builds the outer `Aggregate Data / Epoched Data / … / Encoding` chain, and `cgg_generateEncoderSubFolders_v3` builds the 13-level encoder/classifier subtree:

```matlab
folder = cgg_generateDecodingFolders(baseDir, epoch, target);
folder = cgg_generateEncoderSubFolders_v3(folder, cfg);   % 13 more levels from cfg
% → .../Encoding/Dimension/GRU/'Variational - Stochastic ...'/.../Fold_1/
save(fullfile(folder, 'CM_Table.mat'), 'CM_Table');
```

Each subfolder name is a formatted string of config values (`'Initial Learning Rate - 1.00e-03 ~ ...'`). The MATLAB results aggregator `DATA_cggAllNetworkEncoderResults.m` later walks this tree to find and group every run's `CM_Table.mat` — so the naming must be *exactly* reproducible. The port maps this to `build_matlab_run_dirs(base_dir, cfg)` (Critical Note #15).

## Section 2 — The Python concepts you need

### 2.1 — Why put the config in the path?

There are two schools of thought for organizing ML run outputs:

* **Flat + metadata:** every run gets a random ID (`run_a3f9/`), and the config is stored *inside* as a JSON/YAML file. Simple to write, but you can't tell runs apart without opening each one, and grouping requires a database.
* **Config-encoded path:** the directory *structure* mirrors the config, so `.../Initial Learning Rate - 1.00e-03/.../Fold_1/` tells you everything at a glance, and grouping is just "list the directories at level N." This is MATLAB's choice.

The config-encoded path makes the filesystem itself the index: to find "all runs with LR 1e-3," you glob a path pattern; to compare folds, you list the leaf directories under one config. No database, fully self-documenting. The cost is that the naming must be *deterministic and exact* — which is the whole engineering challenge (§2.4).

### 2.2 — The 19-level hierarchy, live

In [ ]:
from pathlib import Path
from neural_data_decoding.interop.folder_hierarchy_matlab import build_matlab_run_dirs

cfg = {
    "fold": 1, "epoch": "Decision", "target": "Dimension", "model_name": "GRU",
    "is_variational": True, "encoder_output_type": "Stochastic", "activation": "",
    "dropout": 0.5, "want_normalization": False, "bottle_neck_depth": 1,
    "data_width": 100, "window_stride": 50, "start_end_percent": [None, None],
    "normalization": "Channel - Z-Score", "hidden_sizes": [1000, 500, 250],
    "initial_learning_rate": 1e-3, "gradient_threshold": 100.0, "gradient_clip_type": "Global",
    "optimizer": "ADAM", "l2_factor": 1e-4, "mini_batch_size": 100,
    "max_worker_mini_batch_size": 100, "want_stratified_partition": True,
    "std_channel_offset": 0.03, "std_white_noise": 0.015, "std_random_walk": 7e-4,
    "std_time_shift": 100.0, "want_separate_time_shift": True, "subset": True,
    "loss_type_decoder": "MSE", "num_epochs_autoencoder": 0, "prior_proportion": 0.9,
    "rescale_loss_epoch": 0, "weight_reconstruction": 100.0, "weight_classification": 10.0,
    "weight_kl": 1.0, "weight_confidence": 1.0, "confidence_type": ["Trial", "Task"],
    "want_batch_correction": False,
    "dynamic_parameter_set": "Soft Three-Stage Curriculum - Shortened",
    "stitching_and_fusion_layer": "", "classifier_name": "Deep LSTM - Dropout 0.5",
    "classifier_hidden_size": [250, 100, 50], "weighted_loss": "Inverse",
    "multiple_instance_learning_type": "MIL",
}
dirs = build_matlab_run_dirs(base_dir=Path("/results"), cfg=cfg)
for i, part in enumerate(dirs.classifier_fold.relative_to("/results").parts):
    print(f"  level {i:2}:  {part}")

Read the path top to bottom and it's the entire experiment: the epoch (`Decision`), the decoding target (`Dimension`), the model (`GRU`), the variational config, the data windowing, normalization, hidden sizes, the optimizer settings, the mini-batch config, the augmentation strengths, the loss weights, the curriculum regime, the classifier, and finally `Fold_1`. Nineteen levels, each a formatted slice of `cfg`. Nothing is hidden in a metadata file — it's all in the path.

### 2.3 — The three returned directories

In [ ]:
print("classifier_fold  (leaf — gets CM_Table.mat, checkpoints, EncodingParameters.yaml):")
print("   ", dirs.classifier_fold.name, "under .../", dirs.classifier_fold.parent.name)
print()
print("autoencoder_fold (Information/Fold_N — Stage 1 unsupervised weights, if any):")
print("   ", "/".join(dirs.autoencoder_fold.parts[-2:]))
print()
print("encoding_dir     (top-level Encoding/{Target} — the aggregator's discovery root):")
print("   ", "/".join(dirs.encoding_dir.parts[-2:]))

`build_matlab_run_dirs` returns three paths ([`MatlabRunDirs`](../../src/neural_data_decoding/interop/folder_hierarchy_matlab.py)):

* **`classifier_fold`** — the leaf `Fold_{N}/`, where the run writes its `CM_Table.mat` (test-set results, [08.2](08.2_writing_mat_files_for_matlab.ipynb)), `CM_Table_Validation.mat`, `EncodingParameters.yaml`, and checkpoints ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)).
* **`autoencoder_fold`** — the parallel `Information/Fold_{N}/`, holding the Stage 1 unsupervised autoencoder weights when `num_epochs_autoencoder > 0` (the two-stage lifecycle, [05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)).
* **`encoding_dir`** — the top-level `Encoding/{Target}/`, the root the MATLAB aggregator walks down from (Critical Note #15).

### 2.4 — Deterministic, exact naming

In [ ]:
# Same config → same path (idempotent). Different config → path diverges at exactly one level.
dirs_a = build_matlab_run_dirs(base_dir=Path("/results"), cfg=cfg)
dirs_b = build_matlab_run_dirs(base_dir=Path("/results"), cfg={**cfg, "initial_learning_rate": 1e-2})

parts_a = dirs_a.classifier_fold.parts
parts_b = dirs_b.classifier_fold.parts
print("same config → identical path?", dirs_a.classifier_fold == build_matlab_run_dirs(base_dir=Path("/results"), cfg=cfg).classifier_fold)
print()
print("changing ONLY the learning rate diverges the path at exactly one level:")
for i, (a, b) in enumerate(zip(parts_a, parts_b)):
    if a != b:
        print(f"  level {i}: '{a}'")
        print(f"        vs '{b}'")

Two properties make the tree work as an index:

* **Idempotent:** the same config always produces the same path, so re-running lands in the same directory (overwriting, not duplicating) — and the aggregator finds it where it expects.
* **One-param-one-level:** changing a single hyperparameter changes exactly one path segment (here, the learning-rate level). So "all runs sharing everything but LR" are *sibling directories* — the aggregator groups them by listing one level. Note the number formatting: `1.00e-03` vs `1.00e-02`, a fixed scientific-notation format (`_fmt_exp`) that must match MATLAB's `sprintf` exactly, or the sibling grouping breaks.

### 2.5 — Why byte-exact parity matters here

This is a place where the port *must* match MATLAB character-for-character — not for numerical correctness (§ [06.10](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)) but for *discovery*. `DATA_cggAllNetworkEncoderResults.m` finds runs by matching path strings. If Python formatted the learning rate as `0.001` where MATLAB writes `1.00e-03`, the aggregator would silently *miss* the Python run — no error, just absent results. So the formatting helpers (`_fmt_exp`, `_fmt_hidden_size`, etc.) each reproduce a specific MATLAB `sprintf` pattern, pinned by the interop parity tests. This is the same "silent parity loss" hazard as the reconstruction normalization ([06.10 §2.4](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)) — wrong but non-crashing.

## Section 3 — The `neural_data_decoding` implementation

`MatlabRunDirs` and `build_matlab_run_dirs` — the returned paths and the entry point:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/interop/folder_hierarchy_matlab.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("class MatlabRunDirs"))
# print the field declarations
for k in range(i, i + 2):
    print(f"{k + 1:4} | {src[k]}")
j0 = next(j for j, line in enumerate(src) if line.strip() == "classifier_fold: Path")
for k in range(j0, j0 + 3):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `MatlabRunDirs` is a small dataclass of three `Path`s — the leaf `classifier_fold`, the `autoencoder_fold`, and the discovery-root `encoding_dir` (§2.3).
* `build_matlab_run_dirs(*, base_dir, cfg)` reads every field via `cfg.get(key, default)` so a legacy config missing a field still produces a path (a missing field just gets its default segment) — robust to config evolution.
* The `_name_*` helpers (`_name_model_parameters`, `_name_learning`, `_name_loss`, …) each format one path level; the `_fmt_exp`/`_fmt_hidden_size` helpers reproduce the exact MATLAB number formatting (§2.5).
* The function only *builds* the paths — it does **not** `mkdir`. The caller creates whichever leaf it writes to (`mkdir(parents=True, exist_ok=True)`), so building a path is side-effect-free and cheap.
* Critical Note #15: output under these paths is discoverable by `DATA_cggAllNetworkEncoderResults.m` unmodified — the whole point of the byte-exact parity.

## Section 4 — Hands-on exercises

### Exercise 4.1 — read the config off the path

Given only a `classifier_fold` path, extract the model name and the fold number without access to the original config.

In [ ]:
# Reveal:
parts = dirs.classifier_fold.parts
model = parts[parts.index("Encoding") + 2]        # Encoding / {Target} / {ModelName}
fold = dirs.classifier_fold.name                    # Fold_{N}
print(f"model (from path): {model}")
print(f"fold  (from path): {fold}")
print("→ the path is self-documenting — no metadata file needed to identify the run.")

### Exercise 4.2 — the aggregator's grouping

Build three runs differing only in fold and show they're *sibling* directories under a shared parent (how the aggregator groups folds of one config).

In [ ]:
# Reveal:
folds = [build_matlab_run_dirs(base_dir=Path("/results"), cfg={**cfg, "fold": n}) for n in (1, 2, 3)]
parents = {d.classifier_fold.parent for d in folds}
print("distinct parent dirs across folds 1,2,3:", len(parents), "(they share ONE parent)")
print("leaf names:", [d.classifier_fold.name for d in folds])
print("→ folds of one config are siblings; the aggregator lists this parent to gather them.")

## Section 5 — Common errors

### The MATLAB aggregator can't find Python runs

A path-formatting mismatch (§2.5) — the aggregator matches exact strings. Check the number formatting (`1.00e-03` not `0.001`), separator characters (` ~ `), and level ordering against a known-good MATLAB path. The interop parity tests exist to catch this.

### Runs overwrite each other unexpectedly

Two configs that *should* differ produced the same path — a config field isn't being encoded into the hierarchy (or two fields collide). Diff the paths (§2.4); the diverging level tells you which field is (or isn't) captured.

### `build_matlab_run_dirs` succeeds but nothing is written

It only *builds* paths — it doesn't create directories or write files (§3). You must `mkdir(parents=True, exist_ok=True)` and then save. A missing `mkdir` yields a `FileNotFoundError` on the first write, not from the builder.

### A legacy config crashes the builder

Most missing fields fall back to defaults (`cfg.get(key, default)`), but a field that must be a non-empty string (like `epoch`) will raise if absent or blank. Provide the required identity fields (`epoch`, `target`, `model_name`, `fold`).

### The path is absurdly long (OS limits)

The hierarchy is deep and the names are verbose — on some filesystems the total path can exceed length limits. Keep `base_dir` short, or (on Windows) enable long-path support. MATLAB's own layout has the same property; it's inherent to config-encoded paths.

## Section 6 — Further reading

- [`src/neural_data_decoding/interop/folder_hierarchy_matlab.py`](../../src/neural_data_decoding/interop/folder_hierarchy_matlab.py) — the full generator with all `_name_*` helpers.
- [`tests/unit/test_interop.py`](../../tests/unit/test_interop.py) — the path-parity tests.
- [08.2 writing .mat files for MATLAB](08.2_writing_mat_files_for_matlab.ipynb) — what gets written *into* the leaf directory.

Next up: **[08.2 writing .mat files for MATLAB](08.2_writing_mat_files_for_matlab.ipynb)** — producing the `CM_Table.mat` results file that MATLAB scripts consume.